In [13]:
import os
import random
import shutil

# Define dataset paths
dataset_path = "../dataset"  # Original dataset
output_path = "../Vehicles_Datasets_Splite"  # New split dataset
train_ratio = 0.8  # 80% train, 20% validation

# Create train and val directories
train_dir = os.path.join(output_path, "train")
val_dir = os.path.join(output_path, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Iterate over each category (class)
for category in os.listdir(dataset_path):
    category_path = os.path.join(dataset_path, category)
    if os.path.isdir(category_path):  # Ensure it's a directory
        images = os.listdir(category_path)
        random.shuffle(images)  # Shuffle images

        split_idx = int(len(images) * train_ratio)
        train_images = images[:split_idx]
        val_images = images[split_idx:]

        # Create class subdirectories in train and val folders
        train_category_dir = os.path.join(train_dir, category)
        val_category_dir = os.path.join(val_dir, category)
        os.makedirs(train_category_dir, exist_ok=True)
        os.makedirs(val_category_dir, exist_ok=True)

        # Copy images to train and val folders
        for img in train_images:
            source_path = os.path.join(category_path, img)
            destination_path = os.path.join(train_category_dir, img)

            # Check if it's a file before copying
            if os.path.isfile(source_path):
                shutil.copy(source_path, destination_path)
            else:
                print(f"Skipping directory: {source_path}")  # Print a message for skipped directories

        for img in val_images:
            source_path = os.path.join(category_path, img)
            destination_path = os.path.join(val_category_dir, img)

            # Check if it's a file before copying
            if os.path.isfile(source_path):
                shutil.copy(source_path, destination_path)
            else:
                print(f"Skipping directory: {source_path}")  # Print a message for skipped directories

        print(f"Category '{category}' split completed: {len(train_images)} train, {len(val_images)} val")

print("Dataset splitting complete!")

Category 'airplane' split completed: 2000 train, 500 val
Category 'bicycles' split completed: 1688 train, 422 val
Category 'cars' split completed: 2008 train, 502 val
Category 'motorbikes' split completed: 1840 train, 460 val
Category 'ships' split completed: 2344 train, 586 val
Dataset splitting complete!


In [6]:
from PIL import Image
from torch.utils.data import Dataset

class VehicleDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images, organized into class folders.
            transform (callable, optional): Optional transform to be applied on an image.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.image_paths = []
        self.labels = []

        # Get all image paths and corresponding labels
        for label, category in enumerate(self.classes):
            category_path = os.path.join(root_dir, category)
            for image_name in os.listdir(category_path):
                self.image_paths.append(os.path.join(category_path, image_name))
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")  # Open image and convert to RGB

        if self.transform:
            image = self.transform(image)

        return {"pixel_values": image, "label": label}

In [7]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images to 224x224 for MobileNet
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # Normalize to [-1, 1]
])

In [8]:
from torch.utils.data import DataLoader
import os

dataset_path = "../Vehicles_Datasets_Splite"

# Create datasets
train_dataset = VehicleDataset(root_dir=f"{dataset_path}/train", transform=transform)
val_dataset = VehicleDataset(root_dir=f"{dataset_path}/val", transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [9]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout

# Load pre-trained MobileNet model
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers
base_model.trainable = False

# Define the number of classes (output labels)
num_labels = 4  # You have 4 vehicle types: airplane, bicycles, motorbikes, ships

# Build the model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_labels, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenet_1.00_224 (Functional) │ (None, 7, 7, 1024)     │     3,228,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,360,580 (12.82 MB)

 Trainable params: 131,716 (514.52 KB)

 Non-trainable params: 3,228,864 (12.32 MB)

In [10]:
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [11]:
# Define transformations for training and validation datasets
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder('../Vehicles_Datasets_Splite/train', transform=transform)
val_dataset = datasets.ImageFolder('../Vehicles_Datasets_Splite/val', transform=transform)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Number of classes
num_classes = len(train_dataset.classes)


In [12]:
# Load pre-trained MobileNet
model = models.mobilenet_v2(pretrained=True)

# Modify the classifier
model.classifier[1] = nn.Linear(model.last_channel, num_classes)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


e:\I5-GIC\AI\Project\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\I5-GIC\AI\Project\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [13]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [12]:
# Training loop
for epoch in range(10):  # Adjust epochs as needed
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Update metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/10], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {train_accuracy:.2f}%")


Epoch [1/10], Loss: 0.1626, Accuracy: 94.60%
Epoch [2/10], Loss: 0.1040, Accuracy: 96.67%
Epoch [3/10], Loss: 0.0869, Accuracy: 97.29%
Epoch [4/10], Loss: 0.0711, Accuracy: 97.73%
Epoch [5/10], Loss: 0.0751, Accuracy: 97.54%
Epoch [6/10], Loss: 0.0430, Accuracy: 98.57%
Epoch [7/10], Loss: 0.0847, Accuracy: 97.26%
Epoch [8/10], Loss: 0.0488, Accuracy: 98.34%
Epoch [9/10], Loss: 0.0353, Accuracy: 98.74%
Epoch [10/10], Loss: 0.0766, Accuracy: 97.70%


In [14]:
from tqdm import tqdm  # For progress bar

# Switch the model to evaluation mode
model.eval()
correct = 0

# Evaluate on the validation set
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        images = batch[0].to(device)  # Input images
        labels = batch[1].to(device)  # True labels

        # Forward pass
        outputs = model(images)

        # Calculate accuracy
        correct += (outputs.argmax(dim=1) == labels).sum().item()

# Calculate validation accuracy
accuracy = correct / len(val_dataset)
print(f"Validation Accuracy: {accuracy:.4f}")


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 140/140 [03:12<00:00,  1.38s/it]

Validation Accuracy: 0.1494


In [10]:
import os

# Define the folder path
folder_path = '../Trained_MobileNet_Model'

# Create the folder if it doesn't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder created: {folder_path}")
else:
    print(f"Folder already exists: {folder_path}")


Folder already exists: ../Trained_MobileNet_Model


In [11]:
import torch

# Define the save path
folder_path = '../Trained_MobileNet_Model'
os.makedirs(folder_path, exist_ok=True)  # Ensure the folder exists

model_path = os.path.join(folder_path, 'mobilenet_v2.pth')

# Save the model state dictionary
torch.save(model.state_dict(), model_path)
print(f"Model saved to: {model_path}")


NameError: name 'model' is not defined

##Load Model

In [1]:
import os
import torch
from torchvision import models, transforms
from PIL import Image

# Define paths
model_path = '../Trained_MobileNet_Model/mobilenet_v2.pth'  # Path to the saved model
train_dir = "../Vehicles_Datasets_Splite/train"  # Path to your dataset's train directory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Step 1: Extract class names
class_names = sorted(os.listdir(train_dir))  # Get class names from subdirectories
num_classes = len(class_names)  # Number of classes
print(f"Class Names: {class_names}, Number of Classes: {num_classes}")

# Step 2: Define the model architecture
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = torch.nn.Linear(model.last_channel, num_classes)  # Adjust the classifier layer to match num_classes

# Step 3: Load the model checkpoint
state_dict = torch.load(model_path, map_location=device)

# Ensure strict loading to resolve mismatches
model.load_state_dict(state_dict)  # No strict=False needed as the classifier is correctly initialized
model = model.to(device)
model.eval()  # Set to evaluation mode
print("Model loaded successfully and adjusted for the current dataset!")


Class Names: ['airplane', 'bicycles', 'cars', 'motorbikes', 'ships'], Number of Classes: 5
Model loaded successfully and adjusted for the current dataset!


e:\I5-GIC\AI\Project\.venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\I5-GIC\AI\Project\.venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
C:\Users\Lenovo123\AppData\Local\Temp\ipykernel_16792\3203446554.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will 

In [2]:
from torchvision import transforms
from PIL import Image

# Define the same transforms as used during training
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to MobileNet input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalization
])

# Load and preprocess a single test image
test_image_path = '../test_image/1NDP8MFU0Q67.jpg'  # Replace with your test image path
image = Image.open(test_image_path).convert('RGB')  # Ensure it's RGB
input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension


In [3]:
# Perform prediction
with torch.no_grad():
    output = model(input_tensor)  # Forward pass
    predicted_class = output.argmax(dim=1).item()  # Get the class index with the highest score

# Map the predicted class index to the class label
class_labels = class_names  # Assuming you used ImageFolder
predicted_label = class_labels[predicted_class]

print(f"Predicted Class: {predicted_label}")


Predicted Class: airplane
